# Shotloom on a free Kaggle GPU

Open-source cinematic AI video studio (Wan 2.2, Apache-2.0), running on Kaggle's free GPUs.

**One-time setup**
1. Kaggle account → *Settings* → verify your phone number (unlocks GPU + internet).
2. In this notebook: *Settings* menu → **Accelerator: GPU T4 x2** and **Turn on internet**.

**Engines**
- `cinematic` – Wan 2.2 **A14B** + Lightning 4-step, 4-bit GGUF, one expert per T4. Best quality. Animates a **first frame**
  for every AI shot (make keyframes in Gemini / ChatGPT / Midjourney). First run downloads ~33 GB (10–15 min).
- `fast` – Wan 2.2 **TI2V-5B**. Also does text-to-video. First run downloads ~34 GB.

**Modes**
- `studio` – the web app with a private password-protected link. Runs while this tab is open.
- `batch` – upload your project (or `shots.csv` + images) as a dataset, then *Save Version → Save & Run All*:
  renders every shot in the background for up to 12 h, laptop closed. Download `projects/` from *Output*.

In [ ]:
# ---------------- settings ----------------
MODE     = "studio"         # "studio" (web app) or "batch" (background render)
ENGINE   = "cinematic"      # "cinematic" (Wan 2.2 A14B, best) or "fast" (Wan 2.2 5B, also text-to-video)
QUALITY  = "standard"       # batch mode: "draft", "standard" or "final"
ASPECT   = "16:9"           # "16:9" YouTube, "9:16" Shorts/Reels (used when importing a CSV)
STYLE    = ""               # optional text added to every AI prompt
PROJECT  = "my-film"        # project name when importing from a CSV
PASSWORD = "change-me"      # studio login: user 'shotloom' + this password (avoid the default!)
REPO     = "https://github.com/faisal8143/shotloom.git"

In [ ]:
%%capture install_log
# stop a studio left running by an earlier Run All (it would hold the GPUs)
!pkill -f "app.py --share" || true
!pip install -q -U "diffusers>=0.35.0" "transformers>=4.50" accelerate peft gguf ftfy sentencepiece gradio imageio-ffmpeg
# Kaggle ships an old torchao that breaks recent diffusers imports; Shotloom does not use it
!pip uninstall -y -q torchao
# re-runs: leave the folder before replacing it, or git fails in a deleted directory
%cd /kaggle/working
!rm -rf /kaggle/working/shotloom && git clone -q --depth 1 $REPO /kaggle/working/shotloom

In [ ]:
import glob, os, subprocess, sys, torch
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] or "NONE - turn on GPU T4 x2")
if PASSWORD == "change-me" and MODE == "studio":
    print("WARNING: set your own PASSWORD in the settings cell - 'change-me' is public in the repo")
if not os.path.exists("/kaggle/working/shotloom/app.py"):
    print(install_log.stdout[-3000:], install_log.stderr[-3000:])
    raise SystemExit("install failed - see the log above, then run the install cell again")
os.chdir("/kaggle/working/shotloom")
HOME = "/kaggle/working/projects"
os.makedirs(HOME, exist_ok=True)
os.environ.update(SHOTLOOM_HOME=HOME, SHOTLOOM_ENGINE=ENGINE)

def stream(cmd):
    # stream output live so the cell (and the commit log) shows progress
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in p.stdout:
            print(line, end="")
    except KeyboardInterrupt:          # stop button: stop shotloom too, so it frees the GPUs
        p.terminate()
        try:
            p.wait(30)
        except subprocess.TimeoutExpired:
            p.kill()
        print("\nstopped")
        return
    if p.wait() != 0:
        raise SystemExit("shotloom exited with an error - see the log above")

def cli(*args):
    stream([sys.executable, "-u", "-m", "shotloom.cli", *args, "--home", HOME, "--engine", ENGINE])

# input: a Shotloom project folder, or a shots.csv (+ images) uploaded as a dataset
projects = glob.glob("/kaggle/input/**/project.json", recursive=True)
csvs = glob.glob("/kaggle/input/**/*.csv", recursive=True)
if projects:
    src = os.path.dirname(projects[0]); PROJECT = os.path.basename(src)
    print("using project folder:", src)
    cli("status", "--project", PROJECT, "--copy-from", src)
elif csvs:
    folder = os.path.dirname(csvs[0])
    images = os.path.join(folder, "images") if os.path.isdir(os.path.join(folder, "images")) else folder
    print("importing", csvs[0], "images from", images)
    cli("import", "--project", PROJECT, "--csv", csvs[0], "--images", images,
        "--aspect", ASPECT, "--style", STYLE)
else:
    print("No input dataset - studio mode starts with an empty project list.")

In [ ]:
if MODE == "batch":
    cli("batch", "--project", PROJECT, "--quality", QUALITY, "--placeholders")
    cli("build", "--project", PROJECT)
    print("\nDone. Output tab -> projects/" + PROJECT + " (takes/, exports/, project.json)")

In [ ]:
if MODE == "studio":
    # the password goes in through the environment, never a shell, so $ ! " etc. stay intact
    os.environ["SHOTLOOM_PASSWORD"] = PASSWORD
    stream([sys.executable, "-u", "app.py", "--share", "--home", HOME, "--engine", ENGINE])

### Getting your clips back
*Output* panel → `projects/<your project>` → download. Drop that folder into the `projects` folder of Shotloom on your PC
to pick takes and build the final film, or use `exports/` directly.

Kaggle keeps up to 20 GB of output per notebook; a 10-minute film is roughly 1–2 GB of clips.
Each studio run prints a new `gradio.live` link; stop the session (power icon) when done to save GPU hours.